In [1]:
library(tidyverse)


── Attaching core tidyverse packages ──────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
cars_raw = read_csv("serbia_car_sales_price_2024.csv")

Rows: 8413 Columns: 18
── Column specification ────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (11): post_info, car_name, A/C, emission_class, horsepower, color, type_...
dbl  (7): views, favorite, price, year, seats_amount, car_mileage, km, engin...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [3]:
head(cars_raw)

views,favorite,post_info,price,car_name,year,A/C,emission_class,seats_amount,horsepower,color,"car_mileage, km","engine_capacity, cc",type_of_drive,doors,fuel,car_type,gearbox
<dbl>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>
58,0,posted a week ago,1100,Alfa Romeo 11.9,2002,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
55,0,posted 2 days ago,1100,Alfa Romeo 145,2000,manual A/C,Euro 3,5,106 HP (78 kW),gray,158546,1400,front,2/3 doors,petrol + gas,hatchback,"manual, 5 speeds"
136,4,posted 2 weeks ago,950,Alfa Romeo 145,1999,manual A/C,Euro 3,5,105 HP (77 kW),green,337000,1910,front,2/3 doors,diesel,hatchback,"manual, 5 speeds"
209,1,posted 4 weeks ago,500,Alfa Romeo 146,2000,manual A/C,Euro 3,5,120 HP (88 kW),gray,200000,1600,front,4/5 doors,petrol + gas,limousine,"manual, 5 speeds"
61,0,posted 2 days ago,111,Alfa Romeo 147,2024,automatic A/C,NA,5,116 HP (85 kW),gray,280000,1900,front,2/3 doors,diesel,hatchback,"manual, 5 speeds"
196,2,posted 4 weeks ago,2300,Alfa Romeo 147,2008,no A/C,NA,5,105 HP (77 kW),black,188000,1600,front,4/5 doors,petrol + gas,hatchback,"manual, 5 speeds"


In [4]:
str(cars_raw)

spc_tbl_ [8,413 × 18] (S3: spec_tbl_df/tbl_df/tbl/data.frame)
 $ views              : num [1:8413] 58 55 136 209 61 ...
 $ favorite           : num [1:8413] 0 0 4 1 0 2 1 6 16 0 ...
 $ post_info          : chr [1:8413] "posted a week ago" "posted 2 days ago" "posted 2 weeks ago" "posted 4 weeks ago" ...
 $ price              : num [1:8413] 1100 1100 950 500 111 2300 3500 2150 1350 2000 ...
 $ car_name           : chr [1:8413] "Alfa Romeo 11.9" "Alfa Romeo 145" "Alfa Romeo 145" "Alfa Romeo 146" ...
 $ year               : num [1:8413] 2002 2000 1999 2000 2024 ...
 $ A/C                : chr [1:8413] NA "manual A/C" "manual A/C" "manual A/C" ...
 $ emission_class     : chr [1:8413] NA "Euro 3" "Euro 3" "Euro 3" ...
 $ seats_amount       : num [1:8413] NA 5 5 5 5 5 5 5 5 5 ...
 $ horsepower         : chr [1:8413] NA "106 HP (78 kW)" "105 HP (77 kW)" "120 HP (88 kW)" ...
 $ color              : chr [1:8413] NA "gray" "green" "gray" ...
 $ car_mileage, km    : num [1:8413] NA 158546 337000 

In [5]:
summary(cars_raw)

     views            favorite        post_info             price      
 Min.   :    0.0   Min.   :  0.000   Length:8413        Min.   :  100  
 1st Qu.:   61.0   1st Qu.:  0.000   Class :character   1st Qu.: 1600  
 Median :  114.0   Median :  1.000   Mode  :character   Median : 3300  
 Mean   :  308.7   Mean   :  2.672                      Mean   : 4848  
 3rd Qu.:  245.0   3rd Qu.:  3.000                      3rd Qu.: 5950  
 Max.   :27770.0   Max.   :151.000                      Max.   :82000  
                                                                       
   car_name              year          A/C            emission_class    
 Length:8413        Min.   :1960   Length:8413        Length:8413       
 Class :character   1st Qu.:2003   Class :character   Class :character  
 Mode  :character   Median :2006   Mode  :character   Mode  :character  
                    Mean   :2006                                        
                    3rd Qu.:2010                           

In [6]:
problems(cars_raw)

row,col,expected,actual,file
<int>,<int>,<chr>,<chr>,<chr>


In [7]:
cars_raw <- cars_raw %>%
rename(post_date=post_info, price_eur=price, ac=`A/C`, mileage_km=`car_mileage, km`, engine_capacity_cc=`engine_capacity, cc`)

In [8]:
cars_raw <- cars_raw %>%
mutate(across(c(views, favorite, price_eur, year, seats_amount, mileage_km, engine_capacity_cc), as.integer))

Warning message:
"There was 1 warning in `mutate()`.
ℹ In argument: `across(...)`.
Caused by warning:
! NAs introduced by coercion to integer range"


#### Cleaning and formatting post_date column

In [9]:
cars_raw <- cars_raw %>%
mutate(post_date = str_remove(post_date, "^(posted|updated)\\s?"))

In [10]:
sum(is.na(cars_raw$post_date))

cars_raw %>%
distinct(post_date)

[1] 0

post_date
<chr>
a week ago
2 days ago
2 weeks ago
4 weeks ago
5 days ago
6 days ago
3 weeks ago
yesterday
3 days ago


#### Cleaning AC column

In [11]:
sum(is.na(cars_raw$ac))

cars_raw %>%
distinct(ac)

[1] 10

ac
<chr>
NA
manual A/C
automatic A/C
no A/C


In [24]:
cars_raw <- cars_raw %>%
mutate(ac = str_trim(str_remove(ac, "A/C$")))

In [13]:
cars_raw <- cars_raw %>%
mutate(emission_class = str_remove(emission_class, "^Euro\\s?")) %>%
mutate(emission_class = as.integer(emission_class))

In [14]:
cars_raw <- cars_raw %>%
mutate(horsepower = str_extract(horsepower, "^[0-9]+") %>% as.numeric()) %>%
mutate(horsepower = as.integer(horsepower))

In [25]:
cars_raw <- cars_raw %>%
mutate(doors = str_trim(str_remove(doors, "doors$")))

In [16]:
cars_raw %>%
distinct(color)

# Color names are OK

color
<chr>
NA
gray
green
black
white
red
brown
silver
teget


In [17]:
cars_raw %>%
count(gearbox, sort = TRUE)

gearbox,n
<chr>,<int>
"manual, 5 speeds",4645
"manual, 6 speeds",2323
automatic,1164
"manual, 4 speeds",168
semi-automatic,85
"manual, multiple speeds",18
NA,10


In [19]:
cars_raw <- cars_raw %>%
  mutate(
    # Kettéválasztjuk a vessző mentén (ha nincs vessző, a 'speeds_raw' NA lesz)
    speeds_raw = str_split_i(gearbox, ", ", 2),
    gearbox = str_split_i(gearbox, ", ", 1),
    
    
    # Kiszedjük a számot a 'speeds_raw' oszlopból, ha nincs szám, akkor 0
    speeds = str_extract(speeds_raw, "[0-9]+") %>% as.numeric(),
    speeds = ifelse(is.na(speeds), 0, speeds)
  ) %>%
  select(-speeds_raw) %>%
  mutate(speeds = as.integer(speeds))

In [ ]:
cars_raw %>%
count(car_name, sort = TRUE)

In [20]:
cars_raw

views,favorite,post_date,price_eur,car_name,year,ac,emission_class,seats_amount,horsepower,color,mileage_km,engine_capacity_cc,type_of_drive,doors,fuel,car_type,gearbox,speeds
<int>,<int>,<chr>,<int>,<chr>,<int>,<chr>,<int>,<int>,<int>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>
58,0,a week ago,1100,Alfa Romeo 11.9,2002,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,0
55,0,2 days ago,1100,Alfa Romeo 145,2000,manual,3,5,106,gray,158546,1400,front,2/3,petrol + gas,hatchback,manual,5
136,4,2 weeks ago,950,Alfa Romeo 145,1999,manual,3,5,105,green,337000,1910,front,2/3,diesel,hatchback,manual,5
209,1,4 weeks ago,500,Alfa Romeo 146,2000,manual,3,5,120,gray,200000,1600,front,4/5,petrol + gas,limousine,manual,5
61,0,2 days ago,111,Alfa Romeo 147,2024,automatic,NA,5,116,gray,280000,1900,front,2/3,diesel,hatchback,manual,5
196,2,4 weeks ago,2300,Alfa Romeo 147,2008,no,NA,5,105,black,188000,1600,front,4/5,petrol + gas,hatchback,manual,5
133,1,5 days ago,3500,Alfa Romeo 147,2008,automatic,4,5,170,black,255000,1929,front,4/5,diesel,hatchback,manual,6
290,6,6 days ago,2150,Alfa Romeo 147,2008,manual,4,5,90,black,214000,1600,front,4/5,petrol + gas,hatchback,manual,5
2629,16,a week ago,1350,Alfa Romeo 147,2008,automatic,4,5,120,white,315000,1910,front,4/5,diesel,hatchback,manual,6


In [ ]:
list of cars
Alfa Romeo
Audi
BMW
Chevrolet
Chrysler
Citroen
Dacia
Daewoo
Daihatsu
Dodge
Fiat
Ford
Honda
Hyundai
Isuzu
Iveco
Jaguar
Jeep
Kia
Lada
Lancia
Land Rover
Lexus
Maserati
Mazda
Mercedes
MG
Mini
Mitsubishi
Moskwitch
Nissan
Opel
Peugeot
Pontiac
Porsche
Renault
Rover
Saab
Seat
Skoda
Smart
SsangYong
Subaru
Suzuki
Tesla
Toyota
Trabant
UAZ
Volkswagen
Volvo
VW
Wartburg
Zastava


In [26]:
str(cars_raw)

tibble [8,413 × 19] (S3: tbl_df/tbl/data.frame)
 $ views             : int [1:8413] 58 55 136 209 61 196 133 290 2629 89 ...
 $ favorite          : int [1:8413] 0 0 4 1 0 2 1 6 16 0 ...
 $ post_date         : chr [1:8413] "a week ago" "2 days ago" "2 weeks ago" "4 weeks ago" ...
 $ price_eur         : int [1:8413] 1100 1100 950 500 111 2300 3500 2150 1350 2000 ...
 $ car_name          : chr [1:8413] "Alfa Romeo 11.9" "Alfa Romeo 145" "Alfa Romeo 145" "Alfa Romeo 146" ...
 $ year              : int [1:8413] 2002 2000 1999 2000 2024 2008 2008 2008 2008 2007 ...
 $ ac                : chr [1:8413] NA "manual" "manual" "manual" ...
 $ emission_class    : int [1:8413] NA 3 3 3 NA NA 4 4 4 4 ...
 $ seats_amount      : int [1:8413] NA 5 5 5 5 5 5 5 5 5 ...
 $ horsepower        : int [1:8413] NA 106 105 120 116 105 170 90 120 105 ...
 $ color             : chr [1:8413] NA "gray" "green" "gray" ...
 $ mileage_km        : int [1:8413] NA 158546 337000 200000 280000 188000 255000 214000 315000 17

In [27]:
summary(cars_raw)

     views            favorite        post_date           price_eur    
 Min.   :    0.0   Min.   :  0.000   Length:8413        Min.   :  100  
 1st Qu.:   61.0   1st Qu.:  0.000   Class :character   1st Qu.: 1600  
 Median :  114.0   Median :  1.000   Mode  :character   Median : 3300  
 Mean   :  308.7   Mean   :  2.672                      Mean   : 4848  
 3rd Qu.:  245.0   3rd Qu.:  3.000                      3rd Qu.: 5950  
 Max.   :27770.0   Max.   :151.000                      Max.   :82000  
                                                                       
   car_name              year           ac            emission_class 
 Length:8413        Min.   :1960   Length:8413        Min.   :1.000  
 Class :character   1st Qu.:2003   Class :character   1st Qu.:3.000  
 Mode  :character   Median :2006   Mode  :character   Median :4.000  
                    Mean   :2006                      Mean   :3.962  
                    3rd Qu.:2010                      3rd Qu.:5.000  
    